# 論文2（Murakami et al. 2017, Sci. Rep.）の図をつくる

モデルは `model/MAM_paper2.py`（11/24 に `MAM_basic_v1.py` から分けたもの。同じ seed なら v1 と同じ軌跡になる）。

**使い方**
1. このノートブックは `研究/` をカレントディレクトリにして開く。
2. 最初の2つのセルを実行してから、作りたい図のセルだけを実行する。図は `資料用の図/論文2/` に PNG で保存される。
3. 図に使う設定は `PRESET`（1つだけ使う図）と `COMPARE`（比べる図）で選ぶ。

**設定の組（`PRESETS`）**

| 名前 | 中身 | 特徴 |
|---|---|---|
| `"text"` | 文面どおり（α そのまま・floor・ω は式(13)どおり） | floor のせいで、どの条件でも整列する |
| `"murakami"` | α そのまま・round・ω は式どおり | 図S2 の青（63 / 5 / 32%）が合う。赤の整列度が低い |
| `"best"` | 今のベスト：α 2倍・round・ω は式どおり | 図3・図3a・図S3 の多くが合う（赤 0.84 / 緑 0.13 / 青 0.10） |

文面と違うのは「α 2倍」と「round」（検証レポート 0.4 節）。**資料には、どの設定で描いたかを必ず書く。**

**図3 の3つの条件（`CONDITIONS`）：** 青 P=10, α=0.1（ばらばら）、赤 P=40, α=0.2（一方向に進む群れ）、緑 P=40, α=0.5（その場にとどまる塊）。

**かかる時間の目安**（この PC で。1回 1500 ステップが約 30 秒）：スナップショット 2 分、図3b,c 5 分、図3a 7 分、図S2 3 分、図S3 8 分。

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# True にすると、ステップ数とシード数を減らして素早く動作確認できる（図の値は信用しない）
QUICK = False

def steps(n):
    return max(60, n // 10) if QUICK else n

def seeds(n):
    return 1 if QUICK else n

def save(fig, name):
    """図を OUT に保存する（PNG、200 dpi）"""
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("保存:", path)

def tic():
    return time.time()

def toc(t0, what=""):
    print(f"{what} {time.time() - t0:.0f} 秒")

from model.MAM_paper2 import (CrabSwarmPaper2, PRESETS, PRESET_NAMES, CONDITIONS, COLORS, simulate,
                              polarization2, density_paper2)

OUT = os.path.join("資料用の図", "論文2")
os.makedirs(OUT, exist_ok=True)

PRESET = "best"                  # 1つの設定だけで描く図に使う
COMPARE = ["murakami", "best"]   # 設定を比べる図に使う（"text" を足してもよい）

# 論文の値（図を目で読んだもの。検証レポート 3章）
FIG3 = {"青": (0.2, 0.65), "赤": (0.93, 4.0), "緑": (0.35, 3.2)}            # 図3b,c の (整列度, 密度)
FIGS2 = {"青": (63, 5, 32), "赤": (99, 1, 0), "緑": (98, 2, 0)}             # 図S2 の 相互予期 / 追従 / 自由移動（%）
FIGS3 = {0: 0.10, 1: 0.70, 2: 0.93, 3: 0.93, 5: 0.95, 8: 0.55, 10: 0.73}    # 図S3 の整列度（t=1500）


def draw_swarm(ax, m, title=""):
    """個体の位置と、直前の移動（矢印）を描く"""
    x, y = m.position[:, 0], m.position[:, 1]
    d = m.last_displacement
    ax.quiver(x - d[:, 0], y - d[:, 1], d[:, 0], d[:, 1], angles="xy", scale_units="xy", scale=1,
              width=0.005, color="k", alpha=0.6)
    ax.scatter(x, y, s=8, color="tab:red", zorder=3)
    ax.set_xlim(-1, m.S); ax.set_ylim(-1, m.S); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10)

## 1. スナップショット（青・赤・緑）

`T` ステップ後の位置と、直前の1ステップの移動（矢印）。赤は帯の形の群れが一方向に進み、緑は丸い塊がその場にとどまり、青はばらばら（検証レポート 0.3 節）。

In [ ]:
T = steps(1500)

t0 = tic()
fig, axes = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (name, cond) in zip(axes, CONDITIONS.items()):
    m = CrabSwarmPaper2(seed=0, **cond, **PRESETS[PRESET])
    m.run(T)
    pol = polarization2(m.last_displacement)
    draw_swarm(ax, m, f"{name}：P={cond['P']}, α={cond['alpha']}（整列度 {pol:.2f}）")
fig.suptitle(f"論文2：t={T} のスナップショット（{PRESET_NAMES[PRESET]}）")
fig.tight_layout()
save(fig, f"01_スナップショット_{PRESET}.png")
toc(t0, "かかった時間")

## 2. 整列度と密度の時間変化（図3b,c）

論文の図3b,c は1本のシミュレーションの時系列。こちらは `n_seed` 本を細い線で重ね、破線で論文の値を描く。**時系列をシードで先に平均すると、そろう時刻のばらつきのせいでゆっくり上がる線に見える**ので、1本ずつ描く（検証レポート 3.6 節）。

In [ ]:
T, n_seed = steps(1500), seeds(3)

t0 = tic()
ts = {}   # 条件 -> [(pol, dens, rules), ...]
for name, cond in CONDITIONS.items():
    ts[name] = [simulate(T, seed=s, **cond, **PRESETS[PRESET]) for s in range(n_seed)]
toc(t0, "計算")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name in CONDITIONS:
    for i, (pol, dens, _) in enumerate(ts[name]):
        lab = name if i == 0 else None
        ax[0].plot(pol, color=COLORS[name], lw=0.6, alpha=0.8, label=lab)
        ax[1].plot(dens, color=COLORS[name], lw=0.6, alpha=0.8, label=lab)
    ax[0].axhline(FIG3[name][0], color=COLORS[name], ls="--", lw=1)
    ax[1].axhline(FIG3[name][1], color=COLORS[name], ls="--", lw=1)
ax[0].set_ylabel("整列度"); ax[0].set_ylim(0, 1)
ax[1].set_ylabel("密度（周り3×3の個体数）"); ax[1].set_ylim(0, 6)
for x in ax:
    x.set_xlabel("ステップ"); x.legend(fontsize=8)
fig.suptitle(f"論文2：図3b,c（{PRESET_NAMES[PRESET]}、{n_seed} シード。破線は論文の値）")
fig.tight_layout()
save(fig, f"02_図3bc_時間変化_{PRESET}.png")

# 値の表：各シードで t≥1000 を時間平均 → シードの中央値と範囲（検証レポート 3.6 節の比べ方）
start = min(1000, T // 2)
for name in CONDITIONS:
    p = [r[0][start:].mean() for r in ts[name]]; d = [r[1][start:].mean() for r in ts[name]]
    print(f"{name}: 整列度 {np.median(p):.2f} [{min(p):.2f}〜{max(p):.2f}]（論文 {FIG3[name][0]}）、"
          f"密度 {np.median(d):.2f} [{min(d):.2f}〜{max(d):.2f}]（論文 {FIG3[name][1]}）")

## 3. P × α の格子（図3a）

それぞれの P と α で1本回し、`T` ステップの後半を時間平均した整列度と密度を色で描く。

In [ ]:
Ps = [10, 20, 30, 40]
alphas = [0.1, 0.2, 0.3, 0.4, 0.5]
T = steps(1000)

t0 = tic()
grid_pol = np.zeros((len(Ps), len(alphas))); grid_dens = np.zeros_like(grid_pol)
for i, P in enumerate(Ps):
    for j, a in enumerate(alphas):
        pol, dens, _ = simulate(T, P=P, alpha=a, seed=0, **PRESETS[PRESET])
        grid_pol[i, j] = pol[T // 2:].mean(); grid_dens[i, j] = dens[T // 2:].mean()
toc(t0, "計算")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for x, g, lab, vmax in ((ax[0], grid_pol, "整列度", 1), (ax[1], grid_dens, "密度", 6)):
    im = x.imshow(g, origin="lower", cmap="viridis", vmin=0, vmax=vmax, aspect="auto")
    for i in range(len(Ps)):
        for j in range(len(alphas)):
            x.text(j, i, f"{g[i, j]:.2f}", ha="center", va="center", color="w", fontsize=8)
    x.set_xticks(range(len(alphas)), alphas); x.set_yticks(range(len(Ps)), Ps)
    x.set_xlabel("α"); x.set_ylabel("P"); x.set_title(lab)
    fig.colorbar(im, ax=x)
fig.suptitle(f"論文2：図3a（{PRESET_NAMES[PRESET]}、後半の時間平均、seed 0）")
fig.tight_layout()
save(fig, f"03_図3a_格子_{PRESET}.png")

## 4. ルールの割合（図S2）

t=1000〜1500 の 相互予期 / 追従 / 自由移動 の割合。`COMPARE` の設定と論文を並べる。青は「α そのまま」でぴったり合い、「α 2倍」だと相互予期が少し多すぎる（検証レポート 3.7・3.9 節）。

In [ ]:
T = steps(1500)
start = min(1000, T // 2)

t0 = tic()
frac = {}
for pre in COMPARE:
    for name, cond in CONDITIONS.items():
        _, _, rules = simulate(T, seed=0, **cond, **PRESETS[pre])
        frac[(pre, name)] = rules[start:, :3].mean(0) * 100
toc(t0, "計算")

labels, bars = [], []
for name in CONDITIONS:
    labels.append(f"{name}\n論文"); bars.append(FIGS2[name])
    for pre in COMPARE:
        labels.append(f"{name}\n{pre}"); bars.append(frac[(pre, name)])
bars = np.array(bars, float)
fig, ax = plt.subplots(figsize=(1.2 * len(labels) + 1, 4))
bottom = np.zeros(len(labels))
for k, (rule, col) in enumerate(zip(["相互予期", "追従", "自由移動"], ["tab:red", "tab:green", "tab:gray"])):
    ax.bar(range(len(labels)), bars[:, k], bottom=bottom, color=col, label=rule)
    bottom += bars[:, k]
ax.set_xticks(range(len(labels)), labels, fontsize=8)
ax.set_ylabel("割合（%）"); ax.set_ylim(0, 105); ax.legend(fontsize=8, loc="lower right")
ax.set_title("論文2：図S2 ルールの割合（t≥1000 の平均、seed 0）")
fig.tight_layout()
save(fig, "04_図S2_ルールの割合.png")
for (pre, name), v in frac.items():
    print(f"{name} {pre}: " + " / ".join(f"{q:.0f}" for q in v) + f"（論文 {' / '.join(map(str, FIGS2[name]))}）")

## 5. ω を変えたとき（図S3）

赤（P=40, α=0.2）で ω を振り、t=1400〜1500 の整列度を描く。黒い菱形は論文の図S3（t=1500、1本の値と思われる）。食い違いの中心は ω=0（式どおりだと 0.65、論文は 0.10）。ω をどちら側に掛けるかは、ω と 1/ω の入れ替えと同じ（検証レポート 3.10 節）。

In [ ]:
omegas = [0, 0.5, 1, 2, 3, 5, 8, 10]
T, n_seed = steps(1500), seeds(2)
tail = max(1, T // 15)

t0 = tic()
pol_w = np.zeros((len(omegas), n_seed))
for i, w in enumerate(omegas):
    for s in range(n_seed):
        pol, _, _ = simulate(T, omega=w, seed=s, **CONDITIONS["赤"], **PRESETS[PRESET])
        pol_w[i, s] = pol[-tail:].mean()
toc(t0, "計算")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(omegas, pol_w.mean(1), "o-", color="tab:red", label=f"モデル（{PRESET_NAMES[PRESET]}）")
ax.fill_between(omegas, pol_w.min(1), pol_w.max(1), color="tab:red", alpha=0.15)
ax.plot(list(FIGS3), list(FIGS3.values()), "D", color="k", label="論文 図S3")
ax.set_xlabel("ω"); ax.set_ylabel("整列度（最後の 100 ステップ）"); ax.set_ylim(0, 1)
ax.set_title("論文2：図S3（赤 P=40, α=0.2）")
ax.legend(fontsize=8)
fig.tight_layout()
save(fig, f"05_図S3_ω_{PRESET}.png")